## Imports

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from flood_adapt import FloodAdapt
from flood_adapt.config.config import Settings

## Config

In [ ]:
# FloodAdapt databse inputs
DATA_DIR = Path(r"c:\Users\athanasi\Github\Database\Working_Databases\Charleston\4_FloodAdapt\Database")
site = "charleston_beta_release"

# Monte Carlo input parameters for event sequences
timestep = 1  # timestep of Monte Carlo in years
sim_time = 30  # length of simulation in years
no_seq = 2  # number of event sequences to generate
seed = 42  # random seed for reproducibility

# SLR scenario to evaluate damages over time
slr_scenario_name = "NOAA High" # This needs to be a scenario in the FloodAdapt database

# lookup table
ds_impacts = xr.open_dataset("lookup_table_charleston.nc")

In [ ]:
times = 2020 + np.linspace(0, sim_time, int(sim_time / timestep) + 1)
times

## Initialize FloodAdapt Database

In [ ]:
settings = Settings(
    DATABASE_ROOT=DATA_DIR,
    DATABASE_NAME=site,
)
fa = FloodAdapt(database_path=settings.database_path)
slr_years = [fa.interp_slr(slr_scenario=slr_scenario_name, year=time) for time in times]

In [ ]:
ds_impacts

In [ ]:
# ABM Simulation: Household Floodproofing Decisions (integrated logic)
from abm_simulator import ABMSimulator

# Instantiate the simulator (event sequence and damage lookup are handled internally)
abm = ABMSimulator(
    ds_impacts=ds_impacts,
    times=times,
    slr_times=slr_years,
    no_seq=no_seq,
    damage_threshold=0.3,  # You can change this threshold as needed
    seed=seed
)

In [ ]:
# Run the simulation
abm.run("linear")

In [ ]:
abm.plot_event_damage_timeseries(0)

In [ ]:
abm.plot_total_damage_statistics()